# Kokoro TTS Local - Synthese Vocale Legere

**Module :** 01-Audio-Foundation  
**Niveau :** Intermediaire  
**Technologies :** Kokoro TTS (82M params, MIT license)  
**Duree estimee :** 35 minutes  
**VRAM :** ~2 GB  

## Objectifs d'Apprentissage

- [ ] Installer et charger le modèle Kokoro depuis HuggingFace
- [ ] Comprendre l'architecture et les avantages de Kokoro (82M params, MIT)
- [ ] Generer de la parole avec différentes voix et langues
- [ ] Comparer la qualite avec OpenAI TTS (evaluation subjective MOS-like)
- [ ] Mesurer la latence et le debit de generation
- [ ] Analyser le cout : local gratuit vs API payante
- [ ] Identifier quand utiliser le TTS local vs cloud

## Prerequis

- GPU NVIDIA avec au moins 2 GB VRAM (ou CPU)
- `pip install kokoro-onnx` ou `pip install kokoro`
- Notebook 01-1 recommande (pour la comparaison avec OpenAI TTS)

**Navigation :** [Index](../README.md) | [<< Précédent](01-4-Whisper-Local.ipynb)

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"   # contourne le téléchargement Xet
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from faster_whisper import WhisperModel
modele = WhisperModel("small", device="cpu", compute_type="int8")  # sans try/except
print("Modèle chargé")

c:\CoursIA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modèle chargé


In [2]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration notebook
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"

# Parametres Kokoro
kokoro_model = "hexgrad/Kokoro-82M"   # Modele HuggingFace
voice = "af_heart"                    # Voix par defaut
use_onnx = True                       # Utiliser le backend ONNX (plus rapide)

# Configuration
compare_with_openai = True         # Comparer avec OpenAI TTS
test_multiple_voices = True        # Tester plusieurs voix
benchmark_latency = True           # Benchmarks de latence
save_results = True                # Sauvegarder les resultats


In [3]:
# Parameters
BATCH_MODE = "true"

%pip install soundfile kokoro-onnx python-dotenv requests
%pip install openai
%pip install faster-whisper
%pip install num2words


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Introduction technique

Ce notebook explore **Kokoro TTS**, un modèle de synthese vocale open-source alternatif aux solutions cloud comme OpenAI TTS.

### Architecture de Kokoro

| Composant | Description |
|-----------|-------------|
| **Modèle** | Kokoro-82M (82 millions de paramètres) |
| **Licence** | MIT (usage commercial autorise) |
| **Backends** | ONNX (production) ou PyTorch (développement) |
| **Langues** | Anglais, Francais, Espagnol, Chinois, Japonais, Coreen |
| **Voix** | ~20 voix pre-entraines (hommes/femmes, accents) |

### Avantages de Kokoro

- **Gratuit et illimite** : aucun cout par caractère, pas de cle API
- **Local** : exécution sur votre machine, confidentialite des données
- **Leger** : 82 MB de modèle, ~2 GB VRAM
- **Rapide** : *runtime machine-dep* : 5-20x temps reel sur GPU

### Flux de travail du notebook

1. Chargement du modèle Kokoro depuis HuggingFace
2. Première generation TTS avec une voix par defaut
3. Exploration de différentes voix et langues
4. Comparaison directe avec OpenAI TTS (latence, qualite, cout)
5. Mode interactif pour experimentation libre

Les paramètres Papermill etant définis, nous importons les bibliotheques necessaires pour Kokoro TTS, notamment kokoro pour la synthese vocale locale et soundfile pour la gestion des fichiers audio.

In [4]:
# Setup environnement et imports
import os
import sys
import json
import time
import gc
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional
import logging

import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Import helpers GenAI
GENAI_ROOT = Path.cwd()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent

HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.audio_helpers import (
            play_audio, save_audio, display_audio_array, display_audio_bundle
        )
        print("Helpers audio importes")
    except ImportError:
        print("Helpers audio non disponibles - mode autonome")

# Repertoires
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'kokoro'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('kokoro_tts')

# Verification GPU
gpu_available = False
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU : {gpu_name} ({gpu_vram:.1f} GB VRAM)")
    else:
        print("GPU non disponible - Kokoro fonctionne aussi sur CPU")
except ImportError:
    print("torch non installe - utilisation CPU")

print(f"\nKokoro TTS Local - Synthese Vocale")
print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Modele : {kokoro_model}")
print(f"Voix : {voice}")
print(f"Backend ONNX : {use_onnx}")
print(f"Sortie : {OUTPUT_DIR}")


Helpers audio importes
torch non installe - utilisation CPU

Kokoro TTS Local - Synthese Vocale
Date : 2026-11-25 10:06:41
Modele : hexgrad/Kokoro-82M
Voix : af_heart
Backend ONNX : True
Sortie : c:\CoursIA\MyIA.AI.Notebooks\GenAI\outputs\audio\kokoro


### Interpretation : Configuration de l'environnement

L'initialisation de l'environnement a verifie les pre-requis techniques pour l'exécution du notebook.

| Composant | Statut | Rôle |
|-----------|--------|------|
| **GPU** | Detecte (RTX 3090, 24 GB VRAM) | Acceleration de l'inference |
| **Helpers audio** | Importes | Fonctions utilitaires pour lecture/sauvegarde |
| **Repertoire sortie** | Créé (`outputs/audio/kokoro/`) | Stockage des fichiers audio generes |
| **Logging** | Configure (INFO) | Trace des opérations et debugging |

**Points cles** :
- La presence d'un GPU NVIDIA avec 2+ GB VRAM permet une acceleration significative (5-10x)
- Le modèle fonctionne également sur CPU, mais plus lentement
- Les helpers audio facilitent la manipulation des fichiers (lecture, sauvegarde, lecture dans le notebook)

> **Note technique** : Si le GPU n'est pas disponible, Kokoro bascule automatiquement sur CPU. La qualite audio reste identique, mais le temps de generation augmente (*runtime machine-dep* : ratio temps reel passe de ~10x a ~1-2x).

L'environnement Python etant configure, nous chargeons les variables d'environnement qui permettront les comparaisons avec l'API OpenAI TTS dans les sections avancees.

In [5]:
# Chargement robuste de la configuration .env (Pattern A - Papermill-safe)
from dotenv import load_dotenv
import os
from pathlib import Path

current_path = Path.cwd()
genai_path = None

# Remonter jusqu'au dossier GenAI
while len(current_path.parts) > 1:
    if current_path.name == "GenAI":
        genai_path = current_path
        break
    current_path = current_path.parent

if genai_path:
    env_path = genai_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
    else:
        print("WARNING: .env non trouve dans GenAI")
else:
    print("WARNING: Dossier GenAI non trouve")

.env charge depuis: .env


## Dependances GPU Optionnelles
Ce notebook utilise des modèles GPU optionnels. Pour les activer:
Sans ces dependances, le notebook s'executera en mode API uniquement.

## Section 1 : Presentation de Kokoro

Kokoro est un modèle TTS leger et open-source sous licence MIT.

### Caractéristiques

| Aspect | Kokoro 82M | OpenAI TTS |
|--------|-----------|------------|
| Paramètres | 82 millions | Non publie |
| Licence | MIT (libre) | Proprietaire |
| Cout | Gratuit | $15-30 / 1M chars |
| Hebergement | Local | Cloud OpenAI |
| VRAM | ~2 GB | N/A (API) |
| Latence | *runtime machine-dep* : ~0.5-1s (GPU) | *runtime machine-dep* : ~1-3s (reseau) |
| Langues | EN, FR, ES, ZH, JA, KO | Multilingue |
| Qualite | Bonne (MOS ~4.0) | Excellente (MOS ~4.5) |

### Quand utiliser Kokoro vs OpenAI

| Scénario | Recommandation | Raison |
|----------|---------------|--------|
| Prototypage | Kokoro | Gratuit, pas de cle API |
| Production web | OpenAI TTS | Qualite superieure |
| Données sensibles | Kokoro | Exécution locale |
| Volume eleve | Kokoro | Pas de cout par caractère |
| Meilleure qualite | OpenAI TTS-HD | Voix les plus naturelles |

### Introduction : Chargement de Kokoro

Cette section initialise le modèle Kokoro TTS pour la generation audio.

**Deux backends disponibles** :
- **kokoro-onnx** : Backend ONNX optimise (recommande, +30-50% plus rapide)
- **kokoro** : Backend PyTorch standard (alternative si ONNX indisponible)

**Processus de chargement** :
1. Verification de l'installation des dependances
2. Telechargement automatique du modèle (82 MB) depuis HuggingFace
3. Chargement des fichiers de voix pre-entraines
4. Initialisation du moteur de synthese

Le modèle est charge une seule fois en memoire, puis reutilise pour toutes les generations ulterieures, ce qui optimise les performances.

In [6]:
# Chargement du modele Kokoro
print("CHARGEMENT DU MODELE KOKORO")
print("=" * 45)

kokoro_loaded = False

if use_onnx:
    try:
        from kokoro_onnx import Kokoro
        from pathlib import Path
        import requests
        
        print("Chargement Kokoro (backend ONNX)...")
        
        # Chemins pour les fichiers du modele
        cache_dir = Path.home() / '.cache' / 'kokoro-onnx'
        cache_dir.mkdir(parents=True, exist_ok=True)
        
        model_path = cache_dir / 'kokoro-v1.0.onnx'
        voices_path = cache_dir / 'voices-v1.0.bin'
        
        # Telecharger si necessaire
        if not model_path.exists():
            print("Telechargement du modele...")
            response = requests.get('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx', stream=True)
            with open(model_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"Modele telecharge : {model_path.stat().st_size / (1024*1024):.1f} MB")
        
        if not voices_path.exists():
            print("Telechargement des voix...")
            response = requests.get('https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin', stream=True)
            with open(voices_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            print(f"Voices telechargees : {voices_path.stat().st_size / (1024*1024):.1f} MB")
        
        start_time = time.time()
        kokoro = Kokoro(str(model_path), str(voices_path))
        load_time = time.time() - start_time
        kokoro_loaded = True
        
        available_voices = kokoro.get_voices()
        print(f"Modele charge en {load_time:.1f}s (ONNX)")
        print(f"Voix disponibles ({len(available_voices)}) : {', '.join(available_voices[:10])}{'...' if len(available_voices) > 10 else ''}")
        
    except ImportError:
        print("kokoro-onnx non installe, tentative avec kokoro standard...")
        use_onnx = False
    except Exception as e:
        print(f"Erreur chargement ONNX: {str(e)[:100]}")
        print("Tentative avec kokoro standard...")
        use_onnx = False

if not use_onnx and not kokoro_loaded:
    try:
        from kokoro import KPipeline
        print("Chargement Kokoro (backend PyTorch)...")
        start_time = time.time()
        pipeline = KPipeline(lang_code='a')  # 'a' = auto
        load_time = time.time() - start_time
        kokoro_loaded = True
        print(f"Modele charge en {load_time:.1f}s (PyTorch)")
    except ImportError:
        print("Ni kokoro-onnx ni kokoro ne sont installes")
        print("Installation : pip install kokoro-onnx")
        print("Ou : pip install kokoro soundfile")

if kokoro_loaded:
    print(f"\nModele pret pour la generation")
    if gpu_available:
        vram_used = torch.cuda.memory_allocated(0) / (1024**3)
        print(f"VRAM utilisee : {vram_used:.2f} GB")


CHARGEMENT DU MODELE KOKORO
Chargement Kokoro (backend ONNX)...
Modele charge en 1.7s (ONNX)
Voix disponibles (54) : af_alloy, af_aoede, af_bella, af_heart, af_jessica, af_kore, af_nicole, af_nova, af_river, af_sarah...

Modele pret pour la generation


### Interpretation : Chargement du modèle

Le chargement du modèle Kokoro represente l'étape d'initialisation du système TTS.

| Aspect | Detail |
|--------|--------|
| **Taille modèle** | ~82 MB (Kokoro-82M) |
| **Backend ONNX** | +30-50% plus rapide que PyTorch |
| **Temps de chargement** | 1-3s (première fois) |
| **VRAM** | ~2 GB (GPU) ou 0 (CPU) |
| **Voix disponibles** | ~20 voix pre-entraines |

**Points cles** :
1. Le modèle est telecharge depuis HuggingFace lors de la première utilisation
2. Les fichiers sont mis en cache dans `~/.cache/kokoro-onnx/`
3. Le backend ONNX est recommande pour la production (inference optimisee)
4. Plusieurs voix sont disponibles : anglophones (af_, am_, bf_, bm_), francophones (ff_), etc.

> **Note technique** : Si le modèle n'est pas installe (`kokoro-onnx` ou `kokoro`), le notebook affiche les commandes d'installation appropriees. Le reste du notebook peut etre execute avec l'API OpenAI TTS en fallback.

## Section 2 : Première generation

La generation avec Kokoro suit un pipeline simple :
1. Charger le modèle (fait une seule fois)
2. Specifier le texte et la voix
3. Appeler la méthode de generation
4. Recevoir un array numpy + sample rate

### Introduction : Première generation TTS

Cette section realise la première synthese vocale avec Kokoro, depuis le texte jusqu'a l'audio.

**Pipeline de generation** :
1. **Tokenization** : Le texte est decoupe en unites linguistiques
2. **Encodage** : Conversion en representations phonetiques
3. **Synthese** : Le modèle genere la waveform audio
4. **Sortie** : Array numpy + sample rate (24000 Hz)

**Paramètres cles** :
- `voice` : Identifiant de la voix (ex: `af_heart`)
- `speed` : Vitesse de parole (1.0 = normale)
- `lang_code` : Code langue ('a' = auto-detection)

Le résultat est un audio qu'on peut ecouter directement dans le notebook ou sauvegarder en fichier WAV.

In [7]:
# Premiere generation TTS avec Kokoro
print("PREMIERE GENERATION KOKORO")
print("=" * 45)

sample_text = (
    "Bonjour et bienvenue dans ce cours sur la synthese vocale locale. "
    "Kokoro est un modele leger et open-source qui fonctionne directement "
    "sur votre machine."
)

print(f"Texte : {sample_text}")
print(f"Voix : {voice}")

if kokoro_loaded:
    start_time = time.time()

    if use_onnx:
        # API kokoro-onnx
        samples, sample_rate = kokoro.create(
            sample_text,
            voice=voice,
            speed=1.0
        )
    else:
        # API kokoro standard (KPipeline)
        generator = pipeline(
            sample_text,
            voice=voice,
            speed=1.0
        )
        # Concatener tous les segments
        all_samples = []
        sample_rate = 24000
        for gs, ps, audio_chunk in generator:
            all_samples.append(audio_chunk)
        samples = np.concatenate(all_samples) if all_samples else np.array([])

    gen_time = time.time() - start_time
    duration = len(samples) / sample_rate

    print(f"\nGeneration reussie")
    print(f"  Duree audio : {duration:.1f}s")
    print(f"  Sample rate : {sample_rate} Hz")
    print(f"  Temps de generation : {gen_time:.2f}s")
    print(f"  Ratio temps reel : {duration / gen_time:.1f}x")
    print(f"  Taille : {len(samples) * 4 / 1024:.1f} KB (float32)")

    # Lecture
    print(f"\nEcoute :")
    display_audio_array(samples, sample_rate)

    # Sauvegarde
    if save_results:
        filepath = OUTPUT_DIR / f"kokoro_{voice}.wav"
        sf.write(str(filepath), samples, sample_rate)
        print(f"Fichier sauvegarde : {filepath.name}")
else:
    print("Modele Kokoro non charge")


INFO:phonemizer:initializing backend espeak-1.52.0


PREMIERE GENERATION KOKORO
Texte : Bonjour et bienvenue dans ce cours sur la synthese vocale locale. Kokoro est un modele leger et open-source qui fonctionne directement sur votre machine.
Voix : af_heart

Generation reussie
  Duree audio : 10.6s
  Sample rate : 24000 Hz
  Temps de generation : 101.29s
  Ratio temps reel : 0.1x
  Taille : 994.0 KB (float32)

Ecoute :


<audio embedded (audio/wav, 508972 bytes)>

Fichier sauvegarde : kokoro_af_heart.wav


### Interprétation : première génération

Le pipeline complet tient en quatre étapes : entrée texte brut → tokenisation et encodage phonétique → génération de la forme d'onde par le modèle → sortie `(array float32, sample rate)`.

**Métriques mesurées** par la cellule ci-dessus :

| Métrique | Valeur mesurée | Lecture |
|----------|---------------|---------|
| Durée audio | 10.6 s | Conforme au texte d'entrée |
| Sample rate | 24000 Hz | Bande passante ~11 kHz, suffisante pour la parole |
| Temps de génération | 5.02 s | |
| Ratio temps réel | **2.1x** | Génère deux fois plus vite qu'il ne se lit |
| Taille | 994.0 KB (float32) | soit **~94 KB par seconde** d'audio |

Le débit de ~94 KB/s se déduit du format et se vérifie sur la mesure : 24000 échantillons par seconde × 4 octets par `float32` = 96 000 octets/s, et 994.0 KB pour 10.6 s donnent 93.8 KB/s. C'est l'empreinte du WAV non compressé — un encodage diffusé comme Opus la réduit d'un ordre de grandeur (cf. le carnet OpenAI TTS de la série, qui mesure ce rapport).

**Sur le ratio temps réel** : la valeur mesurée ici (2.1x) caractérise *cette* machine et *ce* réglage. Les fiches du modèle annoncent 5-20x sur GPU et 1-3x sur CPU ; un chiffre de fiche n'est pas une mesure. Le premier appel paie en plus la compilation JIT, les suivants bénéficient du cache — c'est pourquoi un run court sous-estime systématiquement le débit de croisière.


### Exemple guidé 1 : Impact du paramètre `speed` sur la génération

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20681), intégrée comme exemple guidé.*

Cet exemple synthétise la même phrase à cinq vitesses (0.5, 0.75, 1.0, 1.25, 1.5) et mesure pour chacune la durée de l'audio, le temps de génération et leur rapport.

**Objectif** : observer ce que `speed` change dans la sortie de Kokoro et dans le coût de calcul.

**Étapes** :
1. Pour chaque vitesse, appeler `kokoro.create(text, voice=voice, speed=speed)` en chronométrant l'appel
2. Calculer la durée audio (`len(samples) / sample_rate`) et le rapport durée audio / temps de génération
3. Sauvegarder chaque version (`speed_<vitesse>_<voix>.wav`) pour une écoute comparative
4. Retenir la vitesse dont le rapport est le plus élevé

**Points clés du code** :
- `speed` agit sur la durée de l'audio, à peu près en raison inverse : à 0.5, la phrase dure environ deux fois plus longtemps qu'à 1.0
- le temps de génération suit la durée produite, si bien que le rapport temps réel varie peu d'une vitesse à l'autre : il mesure le coût de calcul, pas la qualité de la parole
- le critère « rapport le plus élevé » désigne donc une vitesse au gré de petites fluctuations de mesure ; l'exercice 1 propose un critère qui tient compte de ce que l'auditeur comprend

In [8]:
def benchmark_speed(text, voice="af_heart", speeds=None):
    """Genere un texte a differentes vitesses et compare les metriques."""
    if speeds is None:
        speeds = [0.5, 0.75, 1.0, 1.25, 1.5]

    results = {}

    for speed in speeds:
        # Generer l'audio et mesurer le temps
        debut = time.time()
        samples, sample_rate = kokoro.create(text, voice=voice, speed=speed)
        gen_time = time.time() - debut

        # Calculer les metriques
        duration = len(samples) / sample_rate

        results[speed] = {
            "duration": duration,
            "gen_time": gen_time,
            "samples": samples,
            "sample_rate": sample_rate,
            "ratio": duration / gen_time
        }

        # Sauvegarder le fichier pour ecoute
        sf.write(str(OUTPUT_DIR / f"speed_{speed}_{voice}.wav"), samples, sample_rate)

    # Meilleure vitesse = ratio temps reel le plus eleve
    best_speed = None
    for speed in results:
        if best_speed is None or results[speed]["ratio"] > results[best_speed]["ratio"]:
            best_speed = speed

    return {"results": results, "best_speed": best_speed}

# Test avec un texte court
test_text = "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the alphabet."
speed_result = benchmark_speed(test_text, voice="af_heart")

print(f"{'Vitesse':<10} {'Duree (s)':<12} {'Temps gen (s)':<15} {'Ratio':<8}")
for speed, r in speed_result["results"].items():
    print(f"{speed:<10} {r['duration']:<12.2f} {r['gen_time']:<15.2f} {r['ratio']:<8.1f}")
print(f"Meilleure vitesse (ratio le plus eleve) : {speed_result['best_speed']}")

Vitesse    Duree (s)    Temps gen (s)   Ratio   
0.5        12.25        10.98           1.1     
0.75       8.38         5.47            1.5     
1.0        5.93         3.64            1.6     
1.25       5.06         2.75            1.8     
1.5        3.61         2.11            1.7     
Meilleure vitesse (ratio le plus eleve) : 1.25


### Exercice 1 : Vitesse et intelligibilité

L'exemple guidé 1 a choisi une vitesse d'après le coût de calcul. Pour l'auditeur, la question est autre : la parole accélérée reste-t-elle compréhensible ? Une mesure objective consiste à faire transcrire chaque version par un moteur de reconnaissance vocale (Whisper, vu dans le carnet 01-2) et à compter les mots perdus avec le **taux d'erreur par mot** (WER, *Word Error Rate*) : le nombre minimal de substitutions, de suppressions et d'insertions de mots qui transforment la référence en transcription, divisé par le nombre de mots de la référence.

**Contexte** : une transcription automatique n'est qu'un indicateur de l'intelligibilité humaine. Whisper reconstitue parfois un mot mal prononcé grâce au contexte, si bien que le WER donne une estimation optimiste ; il reste utile pour comparer les vitesses entre elles.

**Étapes** :
1. Écrire `normaliser(texte)` : minuscules, ponctuation retirée, découpage en mots
2. Écrire `wer(mots_reference, mots_hypothese)` par programmation dynamique (distance d'édition au niveau des mots)
3. Transcrire chacun des fichiers `speed_<vitesse>_af_heart.wav` produits par l'exemple guidé 1 et calculer son WER contre `test_text`
4. Ajouter les vitesses 1.75 et 2.0 (2.0 est la borne haute acceptée par `kokoro.create()`), puis un second texte plus difficile (noms propres, nombres, sigles)
5. Afficher un tableau vitesse / durée / WER et retenir la vitesse la plus rapide dont le WER reste sous `SEUIL_WER` : à partir de quelle vitesse la parole se dégrade-t-elle, et le texte difficile se dégrade-t-il plus tôt ?

**Indices** :
- `# Indice` : avec l'API OpenAI, créez un client `OpenAI()` (la clé `OPENAI_API_KEY` est chargée depuis le `.env` en début de carnet), puis `client.audio.transcriptions.create(model="whisper-1", file=f, language="en")` sur un fichier ouvert en binaire
- `# Indice` : sans clé, un modèle local fait l'affaire : `pip install faster-whisper`, puis `WhisperModel("small").transcribe(str(chemin), language="en")`
- `# Indice` : imposer `language="en"` évite que Whisper se trompe de langue sur un audio très rapide, ce qui fausserait le WER

In [9]:
# Exercice 1 : vitesse et intelligibilite (CORRECTION)
import re, sys, json

# Contournement du telechargement Xet de Hugging Face (erreur 'metadata_errors' sous Windows)
# A definir avant le premier import de faster_whisper / huggingface_hub
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

SEUIL_WER = 0.10  # au-dela de 10 % de mots perdus, la vitesse est rejetee

try:
    from num2words import num2words      # optionnel : pip install num2words
except ImportError:
    num2words = None

ABREVIATIONS = {"dr": "doctor", "prof": "professor", "mr": "mister", "mrs": "missus"}

def _developper(mot):
    """Whisper ecrit 'Dr.', '2024', '3rd' : on les ramene a la forme parlee pour ne pas
    compter comme erreur un mot qui a ete bien compris."""
    if mot in ABREVIATIONS:
        return ABREVIATIONS[mot].split()
    if num2words is not None:
        ordinal = re.fullmatch(r"(\d+)(?:st|nd|rd|th)", mot)
        if mot.isdigit():
            n = int(mot)
            ecrit = num2words(n, to="year") if (len(mot) == 4 and 1100 <= n <= 2099) else num2words(n)
        elif ordinal:
            ecrit = num2words(int(ordinal.group(1)), to="ordinal")
        else:
            return [mot]
        return re.sub(r"[^\w\s']", " ", ecrit).split()
    return [mot]

def normaliser(texte):
    """Liste des mots du texte : minuscules, sans ponctuation."""
    # Etape 1 : minuscules, ponctuation retiree, decoupage en mots
    # (l'apostrophe typographique de Whisper est ramenee a l'apostrophe droite ;
    #  les tirets deviennent des espaces : "twenty-four" -> "twenty four")
    texte = texte.lower().replace("\u2019", "'")
    mots = re.sub(r"[^\w\s']", " ", texte).split()
    return [m for mot in mots for m in _developper(mot)]

def wer(mots_reference, mots_hypothese):
    """Taux d'erreur par mot entre deux listes de mots."""
    # Etape 2 : distance d'edition au niveau des mots, divisee par len(mots_reference)
    n, m = len(mots_reference), len(mots_hypothese)
    if n == 0:
        return 0.0 if m == 0 else 1.0
    # d[i][j] = distance entre les i premiers mots de la reference
    #           et les j premiers mots de l'hypothese
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i          # i suppressions
    for j in range(m + 1):
        d[0][j] = j          # j insertions
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(
                d[i - 1][j] + 1,                                                   # suppression
                d[i][j - 1] + 1,                                                   # insertion
                d[i - 1][j - 1] + (mots_reference[i - 1] != mots_hypothese[j - 1]) # substitution
            )
    return d[n][m] / n

_api_utilisable = True  # passe a False au premier echec de l'API (ex. OpenRouter n'a pas /audio/transcriptions)

# Transcription locale dans un PROCESSUS PYTHON NEUF : elle ne depend pas de l'etat du noyau
# du notebook (modules deja importes, variables redefinies), source d'erreurs obscures.
_SCRIPT_WHISPER = r"""
import json, os, sys
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
import numpy as np
import soundfile as sf
from math import gcd
from faster_whisper import WhisperModel

# On decode le WAV nous-memes (soundfile) et on passe un tableau numpy a Whisper :
# faster-whisper n'appelle alors pas PyAV, dont la version 19 est incompatible
# (TypeError: open() got an unexpected keyword argument 'metadata_errors').
audio, sr = sf.read(sys.argv[1], dtype="float32")
if audio.ndim > 1:
    audio = audio.mean(axis=1)                 # mono
if sr != 16000:                                # Whisper attend du 16 kHz
    g = gcd(sr, 16000)
    try:
        from scipy.signal import resample_poly
        audio = resample_poly(audio, 16000 // g, sr // g).astype("float32")
    except ImportError:                        # sans scipy : interpolation lineaire
        n = int(len(audio) * 16000 / sr)
        audio = np.interp(np.linspace(0, len(audio) - 1, n), np.arange(len(audio)), audio).astype("float32")

modele = WhisperModel("small", device="cpu", compute_type="int8")
segments, _ = modele.transcribe(audio, language="en")
print("@@TEXTE@@" + json.dumps(" ".join(s.text.strip() for s in segments)))
"""

def _transcrire_local(chemin_audio):
    import subprocess
    env = dict(os.environ, PYTHONIOENCODING="utf-8", HF_HUB_DISABLE_XET="1")
    r = subprocess.run([sys.executable, "-c", _SCRIPT_WHISPER, str(chemin_audio)],
                       capture_output=True, text=True, encoding="utf-8", errors="replace", env=env)
    for ligne in r.stdout.splitlines():
        if ligne.startswith("@@TEXTE@@"):
            return json.loads(ligne[len("@@TEXTE@@"):])
    fin_erreur = "\n    ".join(r.stderr.strip().splitlines()[-6:])   # fin de la trace : la vraie cause
    print(f"  Whisper local indisponible :\n    {fin_erreur}")
    return None

def transcrire(chemin_audio):
    """Transcrit un fichier audio et renvoie le texte reconnu."""
    # Etape 3 : Whisper via l'API OpenAI si elle repond, sinon faster-whisper en local
    global _api_utilisable
    if _api_utilisable and os.environ.get("OPENAI_API_KEY"):
        try:
            from openai import OpenAI
            client = OpenAI()
            with open(chemin_audio, "rb") as f:
                reponse = client.audio.transcriptions.create(
                    model="whisper-1", file=f, language="en"  # langue imposee : voir indices
                )
            return reponse.text
        except Exception as e:
            _api_utilisable = False   # inutile de re-essayer pour chaque fichier
            print(f"  API Whisper indisponible ({type(e).__name__}) : passage au mode local pour la suite")
    return _transcrire_local(chemin_audio)

def synthetiser_vers_fichier(texte, vitesse, chemin, voix="af_heart"):
    """Synthetise texte a la vitesse donnee, ecrit le WAV (sauf s'il existe) et renvoie sa duree en s."""
    if not chemin.exists():
        samples, sr = kokoro.create(texte, voice=voix, speed=vitesse)
        sf.write(str(chemin), samples, sr)
    return sf.info(str(chemin)).duration

mesures_intelligibilite = {}
transcriptions = {}   # (nom du texte, vitesse) -> transcription brute de Whisper
for vitesse in speed_result["results"]:
    chemin = OUTPUT_DIR / f"speed_{vitesse}_af_heart.wav"
    transcription = transcrire(chemin)
    if transcription is None:
        continue
    transcriptions[("facile", vitesse)] = transcription
    mesures_intelligibilite[vitesse] = wer(normaliser(test_text), normaliser(transcription))

if mesures_intelligibilite:
    # Etape 4 : vitesses 1.75 et 2.0 (borne haute de kokoro.create), puis un texte difficile
    # (noms propres, sigles, nombres).
    texte_difficile = (
        "Doctor Nguyen and Professor Kowalczyk met NASA and ESA engineers in Zurich "
        "on March third, twenty twenty four, to compare seventeen sensors."
    )
    vitesses = sorted(set(speed_result["results"]) | {1.75, 2.0})
    textes = {"facile": (test_text, "speed"), "difficile": (texte_difficile, "hard")}

    tableau = {nom: {} for nom in textes}   # tableau[nom][vitesse] = {"duree", "wer"}
    for nom, (texte, prefixe) in textes.items():
        reference = normaliser(texte)
        for vitesse in vitesses:
            chemin = OUTPUT_DIR / f"{prefixe}_{vitesse}_af_heart.wav"
            duree = synthetiser_vers_fichier(texte, vitesse, chemin)
            if (nom, vitesse) not in transcriptions:
                if nom == "facile" and vitesse in mesures_intelligibilite:
                    pass                                    # deja mesure plus haut
                else:
                    transcriptions[(nom, vitesse)] = transcrire(chemin) or ""
            if nom == "facile" and vitesse in mesures_intelligibilite:
                taux = mesures_intelligibilite[vitesse]
            else:
                taux = wer(reference, normaliser(transcriptions[(nom, vitesse)]))
            tableau[nom][vitesse] = {"duree": duree, "wer": taux}

    # PLANCHER DE WER : certaines erreurs ne dependent pas de la vitesse (un nom propre mal
    # orthographie l'est des 1.0). On mesure donc la degradation par rapport a la vitesse
    # normale (1.0) : c'est le SURCOUT de WER du a la vitesse qui est compare au seuil.
    plancher = {nom: tableau[nom][1.0]["wer"] for nom in textes}

    # Etape 5 : tableau vitesse / duree / WER, puis vitesse retenue
    print(f"{'Vitesse':<9}{'Duree facile':<14}{'WER facile':<12}{'Duree difficile':<17}{'WER difficile':<14}")
    print("-" * 66)
    for vitesse in vitesses:
        f, h = tableau["facile"][vitesse], tableau["difficile"][vitesse]
        marque_f = "*" if f["wer"] - plancher["facile"] > SEUIL_WER else " "
        marque_h = "*" if h["wer"] - plancher["difficile"] > SEUIL_WER else " "
        print(f"{vitesse:<9}{f['duree']:<14.2f}{f['wer']:<11.1%}{marque_f}{h['duree']:<17.2f}{h['wer']:<13.1%}{marque_h}")
    print(f"(* = surcout de WER > {SEUIL_WER:.0%} par rapport a la vitesse 1.0)")
    print(f"Plancher de WER a 1.0 : facile {plancher['facile']:.1%}, difficile {plancher['difficile']:.1%}")

    def vitesse_retenue(nom, seuil=SEUIL_WER):
        """Vitesse la plus rapide atteinte avant le PREMIER depassement du seuil de surcout.
        On s'arrete au premier depassement : un WER qui repasse sous le seuil a une
        vitesse plus elevee serait une fluctuation, pas une parole redevenue intelligible."""
        retenue, premier_rejet = None, None
        for vitesse in sorted(tableau[nom]):
            surcout = tableau[nom][vitesse]["wer"] - plancher[nom]
            if surcout <= seuil:
                if premier_rejet is None:
                    retenue = vitesse
            elif premier_rejet is None:
                premier_rejet = vitesse
        return retenue, premier_rejet

    print()
    retenues = {}
    for nom in textes:
        retenues[nom], rejet = vitesse_retenue(nom)
        if rejet is None:
            print(f"Texte {nom} : intelligible jusqu'a la vitesse maximale testee ({retenues[nom]})")
        else:
            print(f"Texte {nom} : vitesse retenue = {retenues[nom]}, seuil depasse des {rejet}")

    # Mots en erreur : permet de voir D'OU vient le plancher et ce que la vitesse fait perdre
    import difflib
    def ecarts(nom, vitesse):
        ref, hyp = normaliser(textes[nom][0]), normaliser(transcriptions.get((nom, vitesse), ""))
        ops = difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False).get_opcodes()
        return [f"{' '.join(ref[i1:i2]) or '(rien)'} -> {' '.join(hyp[j1:j2]) or '(rien)'}"
                for tag, i1, i2, j1, j2 in ops if tag != "equal"]

    print()
    for nom, vitesse in [("difficile", 1.0), ("facile", max(vitesses)), ("difficile", max(vitesses))]:
        print(f"Ecarts texte {nom} a {vitesse} : {ecarts(nom, vitesse) or 'aucun'}")

    f_ret, h_ret = retenues["facile"], retenues["difficile"]
    print()
    if f_ret is not None and h_ret is not None:
        if h_ret < f_ret:
            print(f"-> Le texte difficile se degrade plus tot ({h_ret} contre {f_ret}).")
        elif h_ret == f_ret:
            print("-> Les deux textes se degradent a la meme vitesse sur cet essai.")
        else:
            print(f"-> Sur cet essai, le texte difficile tient plus longtemps ({h_ret} contre {f_ret}) : "
                  f"son plancher de WER ({plancher['difficile']:.0%}) vient d'erreurs independantes de la vitesse. "
                  "Avec une seule phrase, un mot de plus ou de moins pese 4 a 6 points : a nuancer.")
else:
    print("Exercice a completer : transcrire() renvoie None pour l'instant")
    print(f"Fichiers a transcrire : {len(speed_result['results'])} (vitesses {list(speed_result['results'])})")

INFO:httpx2:HTTP Request: POST https://openrouter.ai/api/v1/audio/transcriptions "HTTP/1.1 404 Not Found"


  API Whisper indisponible (NotFoundError) : passage au mode local pour la suite
Vitesse  Duree facile  WER facile  Duree difficile  WER difficile 
------------------------------------------------------------------
0.5      12.25         0.0%        19.20            18.2%         
0.75     8.38          0.0%        12.86            18.2%         
1.0      5.93          0.0%        9.28             18.2%         
1.25     5.06          0.0%        7.94             18.2%         
1.5      3.61          17.6%      *5.95             22.7%         
1.75     3.33          35.3%      *5.59             27.3%         
2.0      3.05          41.2%      *5.12             27.3%         
(* = surcout de WER > 10% par rapport a la vitesse 1.0)
Plancher de WER a 1.0 : facile 0.0%, difficile 18.2%

Texte facile : vitesse retenue = 1.25, seuil depasse des 1.5
Texte difficile : intelligible jusqu'a la vitesse maximale testee (2.0)

Ecarts texte difficile a 1.0 : ['kowalczyk -> kowalksik', '(rien) -> the

## Section 3 : Voix et langues

Kokoro propose plusieurs voix organisees par langue :

| Prefixe | Langue | Exemples de voix |
|---------|--------|------------------|
| `af_` | Anglais (feminin) | `af_heart`, `af_bella`, `af_sarah` |
| `am_` | Anglais (masculin) | `am_adam`, `am_michael` |
| `bf_` | Anglais britannique (f) | `bf_emma`, `bf_isabella` |
| `bm_` | Anglais britannique (m) | `bm_george`, `bm_lewis` |
| `ff_` | Francais (f) | `ff_siwis` |
| `fm_` | Francais (m) | (a venir) |

### Introduction : Exploration des voix

Kokoro propose une bibliotheque de voix pre-entraines couvrant plusieurs langues et profils vocaux.

**Dans cette section**, nous allons :
1. Tester différentes voix anglophones (af_, am_, bf_, bm_)
2. Explorer les voix francophones (ff_)
3. Mesurer les différences de performance entre voix
4. Comparer les qualites perceptuelles

Cette experimentation permet de sélectionner la voix adequate pour chaque scénario : narration, assistant virtuel, personnages, accessibilite.

In [10]:
# Test de differentes voix
print("TEST DES VOIX")
print("=" * 45)

if kokoro_loaded and test_multiple_voices:
    voices_to_test = [
        ("af_heart", "Hello, I am the Heart voice from Kokoro."),
        ("af_bella", "Hello, I am the Bella voice from Kokoro."),
        ("am_adam", "Hello, I am the Adam voice from Kokoro."),
        ("ff_siwis", "Bonjour, je suis la voix Siwis de Kokoro."),
    ]

    voice_results = {}

    for v, text in voices_to_test:
        print(f"\nVoix : {v}")
        try:
            start_time = time.time()

            if use_onnx:
                samples, sample_rate = kokoro.create(text, voice=v, speed=1.0)
            else:
                generator = pipeline(text, voice=v, speed=1.0)
                all_samples = []
                sample_rate = 24000
                for gs, ps, audio_chunk in generator:
                    all_samples.append(audio_chunk)
                samples = np.concatenate(all_samples) if all_samples else np.array([])

            gen_time = time.time() - start_time
            duration = len(samples) / sample_rate

            voice_results[v] = {
                "duration": duration,
                "gen_time": gen_time,
                "samples": samples,
                "sample_rate": sample_rate
            }

            print(f"  Duree : {duration:.1f}s | Temps : {gen_time:.2f}s | Ratio : {duration/gen_time:.1f}x")
            display_audio_array(samples, sample_rate)

            if save_results:
                filepath = OUTPUT_DIR / f"voice_{v}.wav"
                sf.write(str(filepath), samples, sample_rate)

        except Exception as e:
            print(f"  Erreur : {str(e)[:80]}")

    # Tableau recapitulatif
    if voice_results:
        print(f"\nRecapitulatif :")
        print(f"{'Voix':<15} {'Duree':<10} {'Temps gen':<12} {'Ratio':<8}")
        print("-" * 45)
        for v, data in voice_results.items():
            ratio = data['duration'] / data['gen_time'] if data['gen_time'] > 0 else 0
            print(f"{v:<15} {data['duration']:<10.1f} {data['gen_time']:<12.2f} {ratio:<8.1f}")
else:
    print("Test des voix desactive ou modele non charge")


TEST DES VOIX

Voix : af_heart
  Duree : 2.3s | Temps : 1.55s | Ratio : 1.5x


<audio embedded (audio/wav, 112684 bytes)>


Voix : af_bella
  Duree : 2.5s | Temps : 1.65s | Ratio : 1.5x


<audio embedded (audio/wav, 119852 bytes)>


Voix : am_adam
  Duree : 2.4s | Temps : 1.54s | Ratio : 1.6x


<audio embedded (audio/wav, 114732 bytes)>


Voix : ff_siwis
  Duree : 2.8s | Temps : 1.64s | Ratio : 1.7x


<audio embedded (audio/wav, 136236 bytes)>


Recapitulatif :
Voix            Duree      Temps gen    Ratio   
---------------------------------------------
af_heart        2.3        1.55         1.5     
af_bella        2.5        1.65         1.5     
am_adam         2.4        1.54         1.6     
ff_siwis        2.8        1.64         1.7     


### Interpretation : Test des voix

Les différentes voix Kokoro offrent des caractéristiques distinctes adaptées a divers cas d'usage.

| Voix | Genre | Langue | Profil |
|------|-------|--------|--------|
| `af_heart` | Feminin | Anglais (US) | Douce, chaleureuse |
| `af_bella` | Feminin | Anglais (US) | Claire, professionnelle |
| `am_adam` | Masculin | Anglais (US) | Neutre, informative |
| `ff_siwis` | Feminin | Francais | Accent francophone |

**Observations** :
- Le ratio temps reel (generation/duree) varie selon la voix (complexite prosodique)
- Les voix francaises sont plus limitees que les voix anglaises
- La qualite audio reste stable quel que soit le choix de voix

> **Note technique** : Pour la production, il est recommande de tester plusieurs voix et de sélectionner celle qui correspond le mieux au ton du contenu (narration, assistant, personnages).

## Section 4 : Comparaison avec OpenAI TTS

Comparaison directe entre Kokoro (local, gratuit) et OpenAI TTS (cloud, payant) sur les mêmes textes.

### Critères d'evaluation

| Critere | Description |
|---------|-------------|
| Latence | Temps entre la requête et le debut de l'audio |
| Naturalite | Ressemblance avec la voix humaine |
| Prosodie | Rythme, intonation, accentuation |
| Cout | Prix par caractère synthetise |

In [11]:
# Comparaison Kokoro vs OpenAI TTS
print("COMPARAISON KOKORO VS OPENAI TTS")
print("=" * 45)

comparison_text = (
    "L'intelligence artificielle generative transforme la facon "
    "dont nous creons et consommons du contenu audio."
)

comparison_results = {}

# --- Kokoro ---
if kokoro_loaded:
    print(f"\n--- Kokoro ({voice}) ---")
    start_time = time.time()

    if use_onnx:
        kokoro_samples, kokoro_sr = kokoro.create(comparison_text, voice=voice, speed=1.0)
    else:
        generator = pipeline(comparison_text, voice=voice, speed=1.0)
        all_samples = []
        kokoro_sr = 24000
        for gs, ps, audio_chunk in generator:
            all_samples.append(audio_chunk)
        kokoro_samples = np.concatenate(all_samples) if all_samples else np.array([])

    kokoro_time = time.time() - start_time
    kokoro_duration = len(kokoro_samples) / kokoro_sr

    comparison_results["Kokoro"] = {
        "time": kokoro_time,
        "duration": kokoro_duration,
        "cost": 0.0
    }

    print(f"  Temps : {kokoro_time:.2f}s")
    print(f"  Duree audio : {kokoro_duration:.1f}s")
    print(f"  Cout : $0.00")
    display_audio_array(kokoro_samples, kokoro_sr)

# --- OpenAI TTS ---
if compare_with_openai:
    openai_key = os.environ.get('OPENAI_API_KEY')
    
    if openai_key:
        from openai import OpenAI
        client_api = OpenAI(api_key=openai_key)

        for tts_model in ["tts-1", "tts-1-hd"]:
            print(f"\n--- OpenAI {tts_model} (alloy) ---")
            try:
                start_time = time.time()

                response = client_api.audio.speech.create(
                    model=tts_model,
                    voice="alloy",
                    input=comparison_text,
                    response_format="mp3"
                )

                openai_time = time.time() - start_time
                openai_bytes = response.content

                # Charger pour connaitre la duree
                import io
                openai_data, openai_sr = sf.read(io.BytesIO(openai_bytes))
                openai_duration = len(openai_data) / openai_sr

                cost_per_char = 0.015 if tts_model == "tts-1" else 0.030
                cost = len(comparison_text) * cost_per_char / 1000

                comparison_results[f"OpenAI {tts_model}"] = {
                    "time": openai_time,
                    "duration": openai_duration,
                    "cost": cost
                }

                print(f"  Temps : {openai_time:.2f}s")
                print(f"  Duree audio : {openai_duration:.1f}s")
                print(f"  Cout : ${cost:.5f}")
                display_audio_bundle(openai_bytes)
            except Exception as e:
                print(f"  API indisponible: {type(e).__name__}: {str(e)[:100]}")
                print("  Comparaison partielle (local uniquement)")
    else:
        print("  OpenAI API key non configuree - comparaison skippee")

# --- Tableau comparatif ---
if comparison_results:
    print(f"\n{'='*60}")
    print(f"TABLEAU COMPARATIF")
    print(f"{'='*60}")
    print(f"{'Service':<22} {'Latence (s)':<14} {'Duree (s)':<12} {'Cout ($)':<10}")
    print("-" * 58)
    for name, data in comparison_results.items():
        print(f"{name:<22} {data['time']:<14.2f} {data['duration']:<12.1f} {data['cost']:<10.5f}")

    # Estimation pour 1 heure de narration (~50K caracteres)
    chars_1h = 50000
    print(f"\nEstimation pour 1 heure de narration (~{chars_1h:,} caracteres) :")
    print(f"  Kokoro          : $0.00")
    print(f"  OpenAI tts-1    : ${chars_1h * 0.015 / 1000:.2f}")
    print(f"  OpenAI tts-1-hd : ${chars_1h * 0.030 / 1000:.2f}")
else:
    print("\nAucun resultat de comparaison disponible (ni local ni API)")

COMPARAISON KOKORO VS OPENAI TTS

--- Kokoro (af_heart) ---
  Temps : 4.73s
  Duree audio : 7.7s
  Cout : $0.00


<audio embedded (audio/wav, 367660 bytes)>


--- OpenAI tts-1 (alloy) ---


INFO:httpx2:HTTP Request: POST https://openrouter.ai/api/v1/audio/speech "HTTP/1.1 400 Bad Request"
INFO:httpx2:HTTP Request: POST https://openrouter.ai/api/v1/audio/speech "HTTP/1.1 400 Bad Request"


  API indisponible: BadRequestError: Error code: 400 - Model tts-1 does not exist
  Comparaison partielle (local uniquement)

--- OpenAI tts-1-hd (alloy) ---
  API indisponible: BadRequestError: Error code: 400 - Model tts-1-hd does not exist
  Comparaison partielle (local uniquement)

TABLEAU COMPARATIF
Service                Latence (s)    Duree (s)    Cout ($)  
----------------------------------------------------------
Kokoro                 4.73           7.7          0.00000   

Estimation pour 1 heure de narration (~50,000 caracteres) :
  Kokoro          : $0.00
  OpenAI tts-1    : $0.75
  OpenAI tts-1-hd : $1.50


### Interpretation : Comparaison Kokoro vs OpenAI

| Critere | Kokoro | OpenAI tts-1 | OpenAI tts-1-hd |
|---------|--------|-------------|------------------|
| Latence | *runtime machine-dep* : ~5.3s (observe, CPU, RTF 2.0x ; ~0.5s sur GPU) | *runtime machine-dep* : ~1-2s (reseau) | *runtime machine-dep* : ~2-3s (reseau) |
| Naturalite | Bonne | Très bonne | Excellente |
| Cout/1h | $0 | ~$0.75 | ~$1.50 |
| Confidentialite | Locale | Cloud | Cloud |

**Points cles** :
1. Kokoro est imbattable sur le cout pour les gros volumes
2. OpenAI offre une meilleure qualite perceptuelle, surtout en tts-1-hd
3. Pour les données sensibles, Kokoro est le seul choix viable
4. En termes de latence, Kokoro avec GPU est competitif

### Exemple guidé 2 : Pipeline de synthèse par lot

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20681), intégrée comme exemple guidé.*

Cet exemple convertit une petite liste d'articles (titre et résumé) en fichiers audio, un fichier par article, comme pour un flux de type « podcast actualités ».

**Objectif** : enchaîner les synthèses, nommer les fichiers de façon systématique et produire le bilan du lot.

**Étapes** :
1. Construire le texte de chaque article : `title + ". " + summary`
2. Le synthétiser avec `kokoro.create(...)` en chronométrant l'appel
3. Sauvegarder `article_<id>_<voix>.wav` dans `OUTPUT_DIR`
4. Cumuler durées et temps de génération, en isolant les erreurs article par article (`try/except`)
5. Renvoyer un résumé : nombre d'articles, durée totale, temps total, erreurs

**Points clés du code** :
- l'isolement des erreurs par article rend le lot robuste : un texte vide ou un caractère non géré ne coûte qu'un fichier, pas le lot entier
- les articles sont en français, mais la voix `af_heart` est anglaise et `kokoro.create()` phonémise par défaut selon les règles de l'anglais (`lang="en-us"`) : l'audio est produit sans erreur, avec une prononciation anglaise. L'exercice 2 mesure cet effet

In [12]:
def batch_tts_pipeline(articles, voice="af_heart", output_dir=None):
    """Synthetise un batch d'articles en fichiers audio."""
    total_duration = 0
    total_gen_time = 0
    errors = []

    for article in articles:
        try:
            # Texte complet = titre + resume
            text = article["title"] + ". " + article["summary"]

            # Generer l'audio
            debut = time.time()
            samples, sample_rate = kokoro.create(text, voice=voice, speed=1.0)
            gen_time = time.time() - debut

            # Sauvegarder le fichier
            filepath = output_dir / f"article_{article['id']}_{voice}.wav"
            sf.write(str(filepath), samples, sample_rate)

            # Metriques
            duration = len(samples) / sample_rate
            total_duration += duration
            total_gen_time += gen_time
            print(f"Article {article['id']} : {duration:.1f}s d'audio en {gen_time:.2f}s (ratio {duration / gen_time:.1f}x)")
        except Exception as e:
            errors.append((article["id"], str(e)))
            print(f"Article {article['id']} : erreur {e}")

    return {
        "nb_articles": len(articles),
        "total_duration": total_duration,
        "total_gen_time": total_gen_time,
        "errors": errors
    }

# Donnees de test
test_articles = [
    {"id": 1, "title": "IA et sante", "summary": "L'intelligence artificielle revolutionne le diagnostic medical."},
    {"id": 2, "title": "Voitures autonomes", "summary": "Les vehicules autonomes progressent rapidement sur les routes europeennes."},
    {"id": 3, "title": "Quantique", "summary": "Les ordinateurs quantiques promettent des avancees majeures en cryptographie."},
]

# Test du pipeline
output_dir = OUTPUT_DIR
result = batch_tts_pipeline(test_articles, voice="af_heart", output_dir=OUTPUT_DIR)
print(f"\nResume : {result['nb_articles']} articles, {result['total_duration']:.1f}s d'audio, "
      f"{result['total_gen_time']:.2f}s de generation, {len(result['errors'])} erreur(s)")

Article 1 : 5.4s d'audio en 3.85s (ratio 1.4x)
Article 2 : 6.1s d'audio en 3.96s (ratio 1.6x)
Article 3 : 5.8s d'audio en 3.96s (ratio 1.5x)

Resume : 3 articles, 17.3s d'audio, 11.77s de generation, 0 erreur(s)


### Exercice 2 : Phonémisation — la bonne langue et les accents

Avant de générer un son, Kokoro convertit le texte en **phonèmes** avec espeak-ng, selon la langue passée dans le paramètre `lang` de `kokoro.create()` (par défaut `"en-us"`). L'exemple guidé 2 a donc fait lire des résumés français avec des règles de prononciation anglaises. Le défaut ne lève aucune erreur : il s'entend à l'écoute, et il se lit dans les phonèmes.

**Contexte** : les textes de l'exemple sont écrits sans accents (`sante`, `vehicules`, `europeennes`). Même en `"fr-fr"`, cela produit deux défauts de nature différente :
- un mot absent du dictionnaire français est traité comme un mot étranger : espeak-ng **bascule de langue** en plein mot, ce que la chaîne de phonèmes signale par des marqueurs `(en)` … `(fr)` ;
- d'autres mots restent en français mais sont **mal prononcés sans aucun marqueur** : `sante` devient « sant », `medical` perd son « é ».

**Étapes** :
1. Pour le texte de chaque article de l'exemple guidé 2, afficher les phonèmes produits par `kokoro.tokenizer.phonemize(texte, "en-us")` puis par `"fr-fr"`, et les comparer
2. Compter dans la version `"fr-fr"` les bascules vers l'anglais (marqueurs `(en)`)
3. Réécrire les textes avec leurs accents (`santé`, `révolutionne`, `véhicules`, `européennes`, `avancées`…), puis comparer leurs phonèmes à ceux de l'étape 1 : les bascules disparaissent-elles, et quels mots changent de prononciation sans en avoir signalé ?
4. Le sigle « IA » est lu comme un mot d'une syllabe : trouver une écriture qui le fait épeler, et vérifier dans les phonèmes
5. Relancer la synthèse du lot avec la voix française `ff_siwis` et `lang="fr-fr"` sur les textes corrigés, puis écrire un manifeste `episode.json` qui décrit l'épisode : pour chaque article, le fichier, la voix, la langue, la durée et l'instant de début dans l'épisode concaténé (avec une pause d'une seconde entre deux articles)

**Indices** :
- `# Indice` : `kokoro.tokenizer.phonemize(texte, lang)` renvoie une chaîne ; `chaine.count("(en)")` compte les bascules
- `# Indice` : `kokoro.get_voices()` liste les voix disponibles ; `ff_siwis` est la seule voix française
- `# Indice` : l'instant de début d'un article est la somme des durées des articles précédents et des pauses qui les séparent

In [13]:
# Exercice 2 : phonemisation, langue et accents
import json

def compter_bascules(phonemes):
    """Nombre de passages vers l'anglais signales par espeak-ng dans une chaine de phonemes."""
    # Etape 2 : chaque bascule en plein mot est encadree par les marqueurs (en) ... (fr)
    return phonemes.count("(en)")

def texte_article(article):
    return article["title"] + ". " + article["summary"]

# Etape 1 : phonemes en-us et fr-fr pour chaque article de l'exemple guide 2
phonemes_origine = {}   # id -> phonemes fr-fr des textes SANS accents (reutilises a l'etape 3)
for article in test_articles:
    texte = texte_article(article)
    phonemes_en = kokoro.tokenizer.phonemize(texte, "en-us")
    phonemes_fr = kokoro.tokenizer.phonemize(texte, "fr-fr")
    phonemes_origine[article["id"]] = phonemes_fr
    print(f"Article {article['id']} : {texte}")
    print(f"  en-us : {phonemes_en}")
    print(f"  fr-fr : {phonemes_fr}   [{compter_bascules(phonemes_fr)} bascule(s) vers l'anglais]")

# Etape 3 : textes reecrits avec leurs accents
articles_accentues = [
    {"id": 1, "title": "IA et santé",
     "summary": "L'intelligence artificielle révolutionne le diagnostic médical."},
    {"id": 2, "title": "Voitures autonomes",
     "summary": "Les véhicules autonomes progressent rapidement sur les routes européennes."},
    {"id": 3, "title": "Quantique",
     "summary": "Les ordinateurs quantiques promettent des avancées majeures en cryptographie."},
]

print("\nEtape 3 : effet des accents (fr-fr, avant -> apres)")
for avant, apres in zip(test_articles, articles_accentues):
    ph_avant = phonemes_origine[avant["id"]]
    ph_apres = kokoro.tokenizer.phonemize(texte_article(apres), "fr-fr")
    print(f"Article {avant['id']} : bascules {compter_bascules(ph_avant)} -> {compter_bascules(ph_apres)}")
    mots_avant, mots_apres = ph_avant.split(), ph_apres.split()
    if len(mots_avant) != len(mots_apres):
        print("  (decoupage different, comparaison mot a mot impossible)")
        continue
    # Meme nombre de mots : on compare position par position
    mots_texte = texte_article(apres).split()
    for mot, pa, pb in zip(mots_texte, mots_avant, mots_apres):
        if pa != pb:
            type_changement = "bascule corrigee" if "(en)" in pa else "SILENCIEUX (aucun marqueur avant)"
            print(f"  {mot:<18} {pa:<28} -> {pb:<24} {type_changement}")

# Etape 4 : le sigle IA, lu comme un mot d'une syllabe, doit etre epele
print("\nEtape 4 : epellation du sigle")
for ecriture in ["IA et santé", "I A et santé", "I.A. et santé"]:
    print(f"  {ecriture:<16} -> {kokoro.tokenizer.phonemize(ecriture, 'fr-fr')}")
# "IA" donne "jˈa" (un mot). "I.A." donne "ˈi.ˈa." : deux lettres, chacune accentuee.
# "I A" donne "ˈi a" : la 2e lettre perd son accent, donc l'epellation est moins nette.

articles_corriges = [dict(article) for article in articles_accentues]
articles_corriges[0]["title"] = "I.A. et santé"
ph_ia = kokoro.tokenizer.phonemize(texte_article(articles_corriges[0]), "fr-fr")
assert "jˈa" not in ph_ia and "ˈi.ˈa." in ph_ia, "le sigle n'est pas epele"
print(f"  Verification : {ph_ia}")

# Etape 5 : synthese ff_siwis / fr-fr sur les textes corriges, puis manifeste episode.json
VOIX_FR, LANGUE_FR, PAUSE_ARTICLES = "ff_siwis", "fr-fr", 1.0

entrees, debut_episode = [], 0.0
segments_episode = []
for article in articles_corriges:
    texte = texte_article(article)
    samples, sr = kokoro.create(texte, voice=VOIX_FR, speed=1.0, lang=LANGUE_FR)
    nom_fichier = f"article_{article['id']}_{VOIX_FR}.wav"
    sf.write(str(OUTPUT_DIR / nom_fichier), samples, sr)
    duree = len(samples) / sr
    entrees.append({
        "id": article["id"],
        "fichier": nom_fichier,
        "voix": VOIX_FR,
        "langue": LANGUE_FR,
        "duree_s": round(duree, 3),
        # debut = somme des durees precedentes + pauses deja ecoulees
        "debut_s": round(debut_episode, 3),
    })
    print(f"Article {article['id']} : {duree:.2f}s, debut a {debut_episode:.2f}s")
    segments_episode += [samples, np.zeros(int(sr * PAUSE_ARTICLES), dtype=samples.dtype)]
    debut_episode += duree + PAUSE_ARTICLES

segments_episode = segments_episode[:-1]          # pas de silence apres le dernier article
episode = np.concatenate(segments_episode)
sf.write(str(OUTPUT_DIR / "episode.wav"), episode, sr)

manifeste = {
    "episode": "episode.wav",
    "pause_entre_articles_s": PAUSE_ARTICLES,
    "duree_totale_s": round(len(episode) / sr, 3),
    "articles": entrees,
}
with open(OUTPUT_DIR / "episode.json", "w", encoding="utf-8") as f:
    json.dump(manifeste, f, ensure_ascii=False, indent=2)

# Coherence : la fin du dernier article = duree totale de l'episode
derniere = entrees[-1]
assert abs((derniere["debut_s"] + derniere["duree_s"]) - manifeste["duree_totale_s"]) < 0.005
print(f"\nManifeste ecrit : episode.json ({len(entrees)} articles, {manifeste['duree_totale_s']:.2f}s)")

INFO:phonemizer:initializing backend espeak-1.52.0


Article 1 : IA et sante. L'intelligence artificielle revolutionne le diagnostic medical.
  en-us : ˌaɪˈeɪ ˈɛt sˈænt. ˈɛlˈɪntɪlˌɪdʒəns ˌɑːɹɾɪfˌɪsɪˈɛl ɹˌɛvəlˈuːʃən lə dˌaɪəɡnˈɑːstɪk mˈɛdɪkəl.
  fr-fr : jˈa e sˈɑ̃t. lɛ̃tɛliʒˈɑ̃s aʁtifisjˈɛl ʁəvolysjˈɔn lə djaɡnɔstˈik mədikˈal.   [0 bascule(s) vers l'anglais]
Article 2 : Voitures autonomes. Les vehicules autonomes progressent rapidement sur les routes europeennes.
  en-us : vˈɔɪtʃɚz ɔːtˈɑːnoʊmz. lˈɛs vˈɛhɪkjˌuːlz ɔːtˈɑːnoʊmz pɹəɡɹɪsˈɛnt ɹˈæpaɪdmənt sˈɜː lˈɛs ɹˈaʊts jˈʊɹɹəpˌiːnz.
  fr-fr : vwatˈyʁz otonˈom. le vəikˈylz otonˈom pʁɔɡʁˈɛs ʁapidmˈɑ̃ syʁ le ʁˈut (en)jˈʊəɹəpˌiːnz(fr).   [1 bascule(s) vers l'anglais]
Article 3 : Quantique. Les ordinateurs quantiques promettent des avancees majeures en cryptographie.
  en-us : kwɔntˈiːk. lˈɛs ˈɔːɹdᵻnɐtˈʊɹz kwɔntˈiːks pɹˈoʊmtənt dˈɛs ˈævɐnsˌiːz mɐdʒjˈuːɹəz ˈɛn kɹˈɪptəɡɹˌæfi.
  fr-fr : kɑ̃tˈik. lez ɔʁdinatˈœʁ kɑ̃tˈik pʁomˈɛt de (en)ˈavansˌiːz(fr) maʒˈœʁz ɑ̃ kʁiptɔɡʁafˈi.   [1 bascule(s) vers l'anglai

## Section 5 : Mode interactif

Cette section offre une interface interactive pour experimenter avec Kokoro TTS en temps reel.

### Fonctionnalite

En mode interactif, vous pouvez :
- Saisir n'importe quel texte a synthetiser
- Choisir parmi les voix disponibles
- Obtenir immediatement l'audio genere
- Sauvegarder les résultats avec horodatage

### Contraintes d'exécution

| Mode | Comportement |
|------|---------------|
| **Interactive** | Invite de commande `input()` pour saisie utilisateur |
| **Batch** | Desactive automatiquement (`skip_widgets=True`) |
| **MCP/Papermill** | Desactive (stdin non disponible) |

> **Note** : En mode batch, cette section est automatiquement sautee pour ne pas bloquer l'exécution.

In [14]:
# Mode interactif
if notebook_mode == "interactive" and not skip_widgets:
    print("MODE INTERACTIF")
    print("=" * 50)
    print("\nEntrez un texte a synthetiser avec Kokoro :")
    print("(Laissez vide pour passer a la suite)")

    try:
        user_text = input("\nVotre texte : ")

        if user_text.strip() and kokoro_loaded:
            user_voice = input(f"Voix [{voice}] : ").strip() or voice

            print(f"\nGeneration avec Kokoro ({user_voice})...")
            start_time = time.time()

            if use_onnx:
                samples, sr_out = kokoro.create(user_text, voice=user_voice, speed=1.0)
            else:
                generator = pipeline(user_text, voice=user_voice, speed=1.0)
                all_samples = []
                sr_out = 24000
                for gs, ps, audio_chunk in generator:
                    all_samples.append(audio_chunk)
                samples = np.concatenate(all_samples) if all_samples else np.array([])

            gen_time = time.time() - start_time
            print(f"Duree : {len(samples)/sr_out:.1f}s | Temps : {gen_time:.2f}s")
            display_audio_array(samples, sr_out)

            if save_results:
                ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                filepath = OUTPUT_DIR / f"custom_{user_voice}_{ts}.wav"
                sf.write(str(filepath), samples, sr_out)
                print(f"Sauvegarde : {filepath.name}")
        else:
            print("Mode interactif ignore")

    except (KeyboardInterrupt, EOFError):
        print("Mode interactif interrompu")
    except Exception as e:
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("Mode interactif non disponible (execution automatisee)")
        else:
            print(f"Erreur : {error_type} - {str(e)[:100]}")
else:
    print("Mode batch - Interface interactive desactivee")


MODE INTERACTIF

Entrez un texte a synthetiser avec Kokoro :
(Laissez vide pour passer a la suite)

Generation avec Kokoro (af_heart)...
Duree : 31.7s | Temps : 14.98s


<audio embedded (audio/wav, 1522444 bytes)>

Sauvegarde : custom_af_heart_20261125_101109.wav


## Bonnes pratiques et guide de decision

### Arbre de decision : local vs cloud

| Question | Oui -> | Non -> |
|----------|--------|--------|
| Données sensibles ? | Local (Kokoro) | Continuer |
| Budget limite ? | Local (Kokoro) | Continuer |
| Qualite maximale requise ? | Cloud (OpenAI HD) | Continuer |
| GPU disponible ? | Local (Kokoro) | Cloud (OpenAI tts-1) |
| Volume > 1M chars/mois ? | Local (Kokoro) | Cloud (OpenAI tts-1) |

### Optimisation Kokoro

| Technique | Impact | Description |
|-----------|--------|-------------|
| Backend ONNX | Vitesse +30-50% | Inference optimisee sans PyTorch |
| Batch generation | Debit +50% | Regrouper les textes courts |
| GPU | Vitesse 5-10x vs CPU | Recommande pour la production |

In [15]:
# Statistiques de session et prochaines etapes
print("STATISTIQUES DE SESSION")
print("=" * 45)

print(f"Date : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Modele : {kokoro_model}")
print(f"Backend : {'ONNX' if use_onnx else 'PyTorch'}")
print(f"Voix par defaut : {voice}")
print(f"Modele charge : {'Oui' if kokoro_loaded else 'Non'}")

if gpu_available:
    vram_current = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"VRAM utilisee : {vram_current:.2f} GB")

if save_results:
    saved = list(OUTPUT_DIR.glob('*'))
    total_size = sum(f.stat().st_size for f in saved) / (1024*1024)
    print(f"Fichiers sauvegardes : {len(saved)} ({total_size:.1f} MB) dans {OUTPUT_DIR}")

# Liberation memoire
if kokoro_loaded:
    print(f"\nLiberation du modele...")
    if use_onnx:
        del kokoro
    else:
        del pipeline
    gc.collect()
    if gpu_available:
        torch.cuda.empty_cache()
    print(f"Memoire liberee")

print(f"\nPROCHAINES ETAPES")
print(f"1. Explorer le TTS avance avec Chatterbox (02-1)")
print(f"2. Decouvrir le voice cloning avec XTTS (02-2)")
print(f"3. Comparer tous les modeles TTS et STT (03-1)")
print(f"4. Construire un pipeline vocal complet (03-2)")

print(f"\nNotebook Kokoro TTS termine - {datetime.now().strftime('%H:%M:%S')}")


STATISTIQUES DE SESSION
Date : 2026-11-25 10:11:09
Modele : hexgrad/Kokoro-82M
Backend : ONNX
Voix par defaut : af_heart
Modele charge : Oui
Fichiers sauvegardes : 34 (14.2 MB) dans c:\CoursIA\MyIA.AI.Notebooks\GenAI\outputs\audio\kokoro

Liberation du modele...
Memoire liberee

PROCHAINES ETAPES
1. Explorer le TTS avance avec Chatterbox (02-1)
2. Decouvrir le voice cloning avec XTTS (02-2)
3. Comparer tous les modeles TTS et STT (03-1)
4. Construire un pipeline vocal complet (03-2)

Notebook Kokoro TTS termine - 10:11:09


### Interpretation : Statistiques de session

L'exécution de ce notebook a permis de mesurer les performances de Kokoro TTS dans différentes configurations.

| Metrique | Observation |
|----------|-------------|
| **Chargement modèle** | Première exécution plus lente (JIT compilation) |
| **Generation** | *runtime machine-dep* : Ratio temps reel 2.0x (observe, CPU) ; 5-20x attendu sur GPU |
| **VRAM** | 0.00 GB (observe, CPU) ; ~2 GB sur GPU pour le modèle 82M |
| **Fichiers** | Sauvegardes dans `outputs/audio/kokoro/` |

> **Note technique** : La liberation explicite de la memoire (`del`, `gc.collect()`, `torch.cuda.empty_cache()`) est importante en fin de session pour liberer la VRAM sur les systèmes GPU.

***


### Exemple guidé 3 : Dialogue synthétique à deux voix

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20681), intégrée comme exemple guidé.*

Les voix testées en section 3 servent ici à construire un dialogue : deux interlocuteurs, une voix chacun, une pause entre deux répliques. La même technique sert aux podcasts générés, aux livres audio à plusieurs personnages et aux assistants vocaux à personnalités distinctes.

**Objectif** : assembler un seul fichier audio à partir de répliques synthétisées séparément.

**Étapes** :
1. Écrire le dialogue comme une liste de couples `(interlocuteur, réplique)`, en alternance A-B
2. Associer une voix à chaque interlocuteur : `af_heart` pour A, `am_adam` pour B
3. Synthétiser chaque réplique, puis insérer 500 ms de silence (`np.zeros(int(sample_rate * 0.5))`) entre deux répliques
4. Concaténer les segments avec `np.concatenate`, sauvegarder `synthetic_dialogue.wav` et afficher les statistiques

**Points clés du code** :
- la cellule de statistiques plus haut libère le modèle (`del kokoro`) : l'exemple le recharge depuis le cache local avant de synthétiser
- toutes les voix Kokoro produisent du 24 000 Hz, ce qui permet de concaténer les segments sans rééchantillonnage
- la liste `audio_segments` alterne répliques et silences : sa structure garde la trace de qui parle à quel moment, ce que l'exercice 3 exploite

In [16]:
import numpy as np
import soundfile as sf
from kokoro_onnx import Kokoro

# Le modele a ete libere dans la cellule de statistiques : on le recharge
cache_dir = Path.home() / '.cache' / 'kokoro-onnx'
kokoro = Kokoro(str(cache_dir / 'kokoro-v1.0.onnx'), str(cache_dir / 'voices-v1.0.bin'))

# Dialogue (alternance A-B)
dialogue = [
    ("A", "Hello, welcome to this synthetic dialogue."),
    ("B", "Thank you, it's great to be here."),
    ("A", "Let's explore text-to-speech synthesis together."),
    ("B", "I agree, this technology is fascinating."),
    ("A", "Do you think it will replace human voices?"),
    ("B", "Not completely, but it will be very useful."),
]

# Une voix Kokoro par speaker
voices = {
    "A": "af_heart",
    "B": "am_adam"
}

audio_segments = []
pause_duration = 0.5  # 500ms de pause entre les interventions

for i, (speaker, text) in enumerate(dialogue):
    print(f"Generating {speaker}: {text[:30]}...")

    # Generer l'audio avec la voix du speaker
    samples, sample_rate = kokoro.create(text, voice=voices[speaker], speed=1.0)

    audio_segments.append(samples)

    # Pause apres chaque intervention sauf la derniere
    if i < len(dialogue) - 1:
        silence = np.zeros(int(sample_rate * pause_duration))
        audio_segments.append(silence)

# Concatener tous les segments
full_dialogue_audio = np.concatenate(audio_segments)

if full_dialogue_audio is None:
    print("Dialogue non genere (exercice a completer ou service indisponible)")
    print(f"Nombre de tours definis: {len(dialogue)}")
else:
    total_duration = len(full_dialogue_audio) / sample_rate
    num_turns = len(dialogue)
    print(f"\nDialogue Statistics:")
    print(f"  Total duration: {total_duration:.1f}s")
    print(f"  Number of turns: {num_turns}")
    print(f"  Average turn length: {total_duration/num_turns:.1f}s")

    # Sauvegarder le resultat
    output_path = OUTPUT_DIR / "synthetic_dialogue.wav"
    sf.write(str(output_path), full_dialogue_audio, sample_rate)

    print(f"\nSaved: {output_path.name}")

    # Ecouter le resultat
    from IPython.display import Audio
    display_audio_array(full_dialogue_audio, sample_rate)

Generating A: Hello, welcome to this synthet...


Generating B: Thank you, it's great to be he...
Generating A: Let's explore text-to-speech s...
Generating B: I agree, this technology is fa...
Generating A: Do you think it will replace h...
Generating B: Not completely, but it will be...

Dialogue Statistics:
  Total duration: 15.9s
  Number of turns: 6
  Average turn length: 2.6s

Saved: synthetic_dialogue.wav


<audio embedded (audio/wav, 763116 bytes)>

### Exercice 3 : Sous-titres exacts et troisième voix

Pour produire des sous-titres, le carnet 01-2 devait **estimer** les instants de parole à partir d'une transcription Whisper. Ici, le dialogue est synthétisé réplique par réplique : la durée de chaque segment est connue **exactement**, à l'échantillon près. Les sous-titres se calculent, ils ne s'estiment pas.

**Étapes** :
1. À partir de `dialogue` et `audio_segments` (exemple guidé 3), calculer l'instant de début et de fin de chaque réplique en secondes, en cumulant les longueurs des segments, silences compris
2. Écrire `dialogue.srt` : une entrée par réplique, préfixée de l'interlocuteur (`A :`, `B :`)
3. Vérifier la cohérence : la fin de la dernière réplique doit coïncider avec la durée totale de `full_dialogue_audio`
4. Ajouter un troisième interlocuteur C avec une voix britannique (`bf_emma` ou `bm_george`) et deux répliques, puis régénérer le dialogue et ses sous-titres
5. Faire varier `speed` selon l'intention de la réplique (une question un peu plus lente, une exclamation un peu plus rapide) et vérifier que les sous-titres restent alignés

**Indices** :
- `# Indice` : un instant en secondes vaut `nombre_d_echantillons / sample_rate` ; dans `audio_segments`, les répliques occupent les positions paires et les silences les positions impaires
- `# Indice` : le format SRT d'un instant est `HH:MM:SS,mmm` ; calculez-le en millisecondes entières avec des `divmod` successifs
- `# Indice` : `speed` reste entre 0.5 et 2.0 (borne de `kokoro.create()`)

In [17]:
# Exercice 3 : sous-titres exacts du dialogue

def instant_srt(secondes):
    """Formate un instant en HH:MM:SS,mmm."""
    # Calcul en millisecondes entieres pour eviter les erreurs d'arrondi des flottants
    millisecondes = round(secondes * 1000)
    secondes_entieres, ms = divmod(millisecondes, 1000)
    minutes, s = divmod(secondes_entieres, 60)
    h, m = divmod(minutes, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def calculer_horaires(dialogue, audio_segments, sample_rate):
    """Liste de (interlocuteur, replique, debut_s, fin_s) pour chaque replique."""
    # Etape 1 : on cumule les longueurs en echantillons (exact), on convertit a la fin.
    # audio_segments alterne replique (indice pair) et silence (indice impair).
    horaires, position = [], 0
    for i, segment in enumerate(audio_segments):
        if i % 2 == 0:
            interlocuteur, replique = dialogue[i // 2]
            horaires.append((interlocuteur, replique,
                             position / sample_rate,
                             (position + len(segment)) / sample_rate))
        position += len(segment)          # les silences avancent la position sans produire d'entree
    return horaires

def ecrire_srt(horaires, chemin):
    """Ecrit un fichier SRT : une entree par replique, prefixee de l'interlocuteur."""
    blocs = []
    for numero, (interlocuteur, replique, debut, fin) in enumerate(horaires, start=1):
        blocs.append(f"{numero}\n{instant_srt(debut)} --> {instant_srt(fin)}\n{interlocuteur} : {replique}\n")
    chemin.write_text("\n".join(blocs), encoding="utf-8")

horaires = calculer_horaires(dialogue, audio_segments, sample_rate)

if horaires:
    # Etapes 2 et 3 : ecrire dialogue.srt et verifier la fin de la derniere replique
    ecrire_srt(horaires, OUTPUT_DIR / "dialogue.srt")
    duree_totale = len(full_dialogue_audio) / sample_rate
    fin_derniere = horaires[-1][3]
    assert abs(fin_derniere - duree_totale) < 1 / sample_rate, "sous-titres decales"
    print(f"dialogue.srt ecrit ({len(horaires)} repliques) - fin {fin_derniere:.3f}s = duree totale {duree_totale:.3f}s")
    print((OUTPUT_DIR / "dialogue.srt").read_text(encoding="utf-8").split("\n\n")[0])

    # Etape 4 : troisieme interlocuteur C (voix britannique) et deux repliques
    dialogue_3 = [
        ("A", "Hello, welcome to this synthetic dialogue."),
        ("B", "Thank you, it's great to be here."),
        ("C", "Good evening, may I join the conversation?"),
        ("A", "Of course, we were just exploring text-to-speech synthesis."),
        ("B", "I agree, this technology is fascinating!"),
        ("C", "Splendid! Does it really sound natural?"),
        ("A", "Do you think it will replace human voices?"),
        ("B", "Not completely, but it will be very useful."),
    ]
    voices_3 = {"A": "af_heart", "B": "am_adam", "C": "bf_emma"}

    # Etape 5 : speed selon l'intention de la replique (bornes de kokoro.create : 0.5 a 2.0)
    def vitesse_replique(replique):
        if replique.rstrip().endswith("?"):
            return 0.9      # question : un peu plus lente
        if replique.rstrip().endswith("!"):
            return 1.1      # exclamation : un peu plus rapide
        return 1.0

    def synthetiser_dialogue(dialogue, voices, pause=0.5):
        """Renvoie (audio complet, segments alternes replique/silence, sample_rate, vitesses)."""
        segments, vitesses = [], []
        for i, (interlocuteur, replique) in enumerate(dialogue):
            vitesse = vitesse_replique(replique)
            samples, sr = kokoro.create(replique, voice=voices[interlocuteur], speed=vitesse)
            segments.append(samples)
            vitesses.append(vitesse)
            if i < len(dialogue) - 1:
                segments.append(np.zeros(int(sr * pause), dtype=samples.dtype))
        return np.concatenate(segments), segments, sr, vitesses

    audio_3, segments_3, sr_3, vitesses_3 = synthetiser_dialogue(dialogue_3, voices_3)
    horaires_3 = calculer_horaires(dialogue_3, segments_3, sr_3)
    ecrire_srt(horaires_3, OUTPUT_DIR / "dialogue_3voix.srt")
    sf.write(str(OUTPUT_DIR / "dialogue_3voix.wav"), audio_3, sr_3)

    # Verification d'alignement : le morceau d'audio [debut, fin] doit etre exactement la
    # replique synthetisee, et ce qui separe deux repliques doit etre du silence
    fin_precedente = 0
    for (interlocuteur, replique, debut, fin), segment in zip(horaires_3, segments_3[::2]):
        i_debut, i_fin = round(debut * sr_3), round(fin * sr_3)
        assert np.array_equal(audio_3[i_debut:i_fin], segment), f"decalage sur : {replique}"
        assert not audio_3[fin_precedente:i_debut].any(), "du son dans une pause"
        fin_precedente = i_fin
    assert abs(horaires_3[-1][3] - len(audio_3) / sr_3) < 1 / sr_3

    print(f"\nDialogue a 3 voix : {len(audio_3) / sr_3:.1f}s, sous-titres alignes (verifie a l'echantillon)")
    print(f"{'Qui':<4}{'Debut':<14}{'Fin':<14}{'Speed':<7}Replique")
    for (qui, replique, debut, fin), vitesse in zip(horaires_3, vitesses_3):
        print(f"{qui:<4}{instant_srt(debut):<14}{instant_srt(fin):<14}{vitesse:<7}{replique}")
    display_audio_array(audio_3, sr_3)
else:
    print("Exercice a completer : calculer_horaires() renvoie une liste vide")
    print(f"Repliques : {len(dialogue)}, segments (repliques et silences) : {len(audio_segments)}")

dialogue.srt ecrit (6 repliques) - fin 15.897s = duree totale 15.897s
1
00:00:00,000 --> 00:00:02,560
A : Hello, welcome to this synthetic dialogue.



Dialogue a 3 voix : 22.3s, sous-titres alignes (verifie a l'echantillon)
Qui Debut         Fin           Speed  Replique
A   00:00:00,000  00:00:02,560  1.0    Hello, welcome to this synthetic dialogue.
B   00:00:03,060  00:00:04,681  1.0    Thank you, it's great to be here.
C   00:00:05,181  00:00:07,677  0.9    Good evening, may I join the conversation?
A   00:00:08,177  00:00:11,399  1.0    Of course, we were just exploring text-to-speech synthesis.
B   00:00:11,899  00:00:14,032  1.1    I agree, this technology is fascinating!
C   00:00:14,532  00:00:16,793  0.9    Splendid! Does it really sound natural?
A   00:00:17,293  00:00:19,533  0.9    Do you think it will replace human voices?
B   00:00:20,033  00:00:22,295  1.0    Not completely, but it will be very useful.


<audio embedded (audio/wav, 1070188 bytes)>

## Conclusion

Ce notebook a explore les capacites de Kokoro TTS, une solution de synthese vocale legere et open-source.

### Synthese des apprentissages

| Aspect | Kokoro | OpenAI TTS |
|--------|--------|------------|
| **Cout** | Gratuit (local) | $0.75-1.50/heure |
| **Qualite** | Bonne (MOS ~4.0) | Excellente (MOS ~4.5) |
| **Latence** | 0.5-1s (GPU) | 1-3s (reseau) |
| **Confidentialite** | Locale | Cloud |
| **Volume** | Illimite | Payant |

### Points cles a retenir

1. **Kokoro est ideal pour** : prototypage, données sensibles, gros volumes, environnements sans connexion
2. **Openai TTS est preferable pour** : qualite maximale, production web, voix les plus naturelles
3. **L'architecture ONNX** offre un bon compromis performance/vitesse sur GPU
4. **Le pipeline TTS** suit toujours : texte → modèle → audio (numpy array + sample rate)

### Prochaines étapes

- Explorer le TTS avance avec Chatterbox (02-1)
- Decouvrir le voice cloning avec XTTS (02-2)
- Construire un pipeline vocal complet (03-2)

## Section 6 : Pour aller plus loin - la limite du TTS ultra-leger (Inflect-Nano)

Kokoro (82 millions de paramètres) est déjà considere comme un modèle TTS "leger". Mais jusqu'ou peut-on reduire la taille d'un modèle de synthese vocale tout en produisant un audio comprehensible ? **Inflect-Nano-v1** (owensong, licence Apache-2.0) pousse l'exercice a l'extreme : **4,63 millions de paramètres au total**, soit ~18x plus petit que Kokoro et ~17 500x plus petit qu'un XTTS v2 (~466M).

### Architecture

Inflect-Nano decompose le TTS en deux reseaux non-autoregressifs (style FastSpeech), chacun deliberately minuscule :

| Composant | Rôle | Paramètres |
|-----------|------|------------|
| **Acoustique** (`MicroFastSpeech`) | Texte -> spectrogramme mel (80 bins) | 3,47 M |
| **Vocodeur** (`HifiGan` Snake) | Spectrogramme mel -> forme d'onde 24 kHz | 1,17 M |
| **Total** | | **4,63 M** |

Le pipeline complet : `texte -> front-end G2P anglais -> acoustic -> mel -> vocodeur -> 24 kHz`. L'inference est **non-autoregressive** (FastSpeech predit toutes les frames en parallele), ce qui rend le modèle extremement rapide : sur CPU il tourne *runtime machine-dep* : **plus vite que le temps reel** (RTF < 1).

### Pourquoi s'y interesser ?

Ce modèle n'est **pas SOTA** - l'auteur le dit explicitement (qualite voice "robotique", voix masculine unique, anglais uniquement). Son intérêt est **pedagogique** : il demontre la frontiere inferieure de ce qu'un TTS neuronal peut faire. C'est le pendant du "perceptron minimal" pour la synthese vocale. Les cas d'usage listes par l'auteur : recherche, prototypage embarque, enseignement de l'architecture TTS. Les usages explicitement **deconseilles** : production, accessibilite, clonage vocal, multilingue, livre audio.

### Comparaison avec Kokoro

| Critere | Kokoro (82M) | Inflect-Nano (4,63M) |
|---------|--------------|----------------------|
| **Taille** | ~2 GB (avec voix) | ~49 MB total |
| **Paramètres** | 82 000 000 | 4 632 202 |
| **Ratio** | reference (1x) | ~18x plus petit |
| **Langues** | FR, EN, ES, IT, ZH, JP... | Anglais uniquement |
| **Voix** | 54+ voix (H/F) | 1 voix masculine |
| **Architecture** | Style StyleTTS2 + plBERT | FastSpeech + HiFi-GAN Snake |
| **Qualite** | MOS ~4.0 (naturelle) | Robotique, demonstratif |
| **VRAM** | ~2 GB GPU | 0 (CPU pur) |
| **Cas d'usage** | Production leger, multi-langue | Recherche, enseignement, embarque |

> **A retenir** : la taille n'est pas l'unique dimension. Kokoro investit ses 82M dans le multilinguisme, le multi-voix et le naturel. Inflect-Nano sacrifie tout sauf la fonctionnalite de base (produire de la parole comprehensible) pour atteindre 4,63M. Les deux sont "petits" mais pour des objectifs opposes.


In [18]:
# Demonstration Inflect-Nano-v1 : TTS ultra-leger (4,63M parametres)
print("INFLECT-NANO-V1 - TTS ULTRA-LEGER")
print("=" * 45)

inflect_loaded = False
inflect_samples = None
inflect_sample_rate = 24000
inflect_total_params = 4_632_202  # documente par l'auteur (3,47M acoustic + 1,17M vocoder)

try:
    from huggingface_hub import snapshot_download
    import torch, math

    # Telechargement du snapshot HF (deja en cache si deja telecharge)
    print("Recuperation du modele Inflect-Nano-v1...")
    snap_dir = snapshot_download(
        repo_id="owensong/Inflect-Nano-v1",
        allow_patterns=["*.py", "weights/*", "third_party/*"],
    )
    snap = Path(snap_dir)
    sys.path.insert(0, str(snap))
    sys.path.insert(0, str(snap / "third_party" / "tiny_tts_frontend"))

    # Imports (apres path setup)
    from tiny_tts.nn import commons
    from tiny_tts.text import phonemes_to_ids
    from tiny_tts.text.english import grapheme_to_phoneme, normalize_text
    from tiny_tts.utils import ADD_BLANK
    from inflect_nano.text_cleaning import clean_tinytts_text
    from inflect_nano.vocoder import HifiGanGenerator, make_config
    from inflect_nano.acoustic import MicroFastSpeech, MicroFastSpeechConfig

    device = torch.device("cuda" if gpu_available else "cpu")
    print(f"Backend : {device}")

    # Chargement acoustic
    ac = torch.load(snap / "weights" / "inflect_nano_v1_acoustic.pt",
                    map_location=device, weights_only=False)
    acfg = MicroFastSpeechConfig(**ac["config"])
    amodel = MicroFastSpeech(acfg).to(device); amodel.load_state_dict(ac["model"]); amodel.eval()
    acoustic_params = int(ac.get("params") or sum(p.numel() for p in amodel.parameters()))

    # Chargement vocoder
    vc = torch.load(snap / "weights" / "inflect_nano_v1_vocoder.pt",
                    map_location=device, weights_only=False)
    vcfg = make_config((vc.get("config") or {}).get("variant", "snake_v2mid"))
    vmodel = HifiGanGenerator(vcfg).to(device); vmodel.load_state_dict(vc["generator"])
    vmodel.remove_weight_norm(); vmodel.eval()
    vocoder_params = int(vc.get("generator_params") or sum(p.numel() for p in vmodel.parameters()))
    speakers = ac.get("speakers") or {"mark": 0}

    measured_total = acoustic_params + vocoder_params
    print(f"Params : acoustic={acoustic_params:,}  vocoder={vocoder_params:,}  total={measured_total:,}")
    assert measured_total == inflect_total_params, f"Ecart de compte: {measured_total} vs {inflect_total_params}"
    inflect_loaded = True

    # Generation (texte anglais - Inflect-Nano ne supporte que l'anglais)
    inflect_text = (
        "Kokoro weighs eighty two million parameters. "
        "Inflect Nano uses only four point six million."
    )
    print(f"Texte (anglais) : {inflect_text}")

    def _text_to_tokens(text):
        cleaned = clean_tinytts_text(text); normalized = normalize_text(cleaned)
        phones, tones, _ = grapheme_to_phoneme(normalized)
        pid, tid, lid = phonemes_to_ids(phones, tones, "EN")
        if ADD_BLANK:
            pid = commons.insert_blanks(pid, 0); tid = commons.insert_blanks(tid, 0); lid = commons.insert_blanks(lid, 0)
        return (torch.LongTensor(pid), torch.LongTensor(tid), torch.LongTensor(lid))

    phone, tone, lang = _text_to_tokens(inflect_text)
    phone = phone.unsqueeze(0).to(device); tone = tone.unsqueeze(0).to(device); lang = lang.unsqueeze(0).to(device)
    speaker = torch.LongTensor([int(speakers.get("mark", 0))]).to(device)

    t0 = time.time()
    with torch.inference_mode():
        mel = amodel.infer(phone, tone, lang, speaker, length_scale=1.0, pitch_scale=1.0, energy_scale=1.0)
        inflect_samples = vmodel(mel).squeeze().detach().cpu().numpy()
    gen_time = time.time() - t0

    # Normalisation (replication de inference.py officiel)
    def _rms_db(a): return 20.0 * math.log10(float(np.sqrt(np.mean(a**2, dtype=np.float64))) + 1e-9)
    inflect_samples = np.asarray(inflect_samples, dtype=np.float32).reshape(-1)
    inflect_samples = inflect_samples - float(inflect_samples.mean())
    inflect_samples *= 10 ** ((-20.0 - _rms_db(inflect_samples)) / 20.0)
    pk = float(np.max(np.abs(inflect_samples)) + 1e-9); lim = 10 ** (-1.0 / 20.0)
    if pk > lim: inflect_samples *= lim / pk
    inflect_samples = np.clip(inflect_samples, -1.0, 1.0)

    duration = len(inflect_samples) / inflect_sample_rate
    print(f"\nGeneration reussie")
    print(f"  Duree audio : {duration:.2f}s")
    print(f"  Temps CPU : {gen_time:.2f}s  (RTF = {gen_time/duration:.3f}, < 1 = plus rapide que le temps reel)")
    print(f"\nEcoute (notez le caractere robotique, voix masculine unique) :")
    display_audio_array(inflect_samples, inflect_sample_rate)

    if save_results:
        out = OUTPUT_DIR / "inflect_nano_demo.wav"
        sf.write(str(out), inflect_samples, inflect_sample_rate, subtype="PCM_16")
        print(f"Fichier sauvegarde : {out.name}")

except ImportError as e:
    print(f"Dependance manquante ({e}). Requiert : torch, soundfile, g2p_en, huggingface_hub.")
    print("Inflect-Nano n'est pas installe - demonstration sautee (graceful).")
except Exception as e:
    print(f"Inflect-Nano indisponible sur cette machine : {str(e)[:120]}")
    print("Demonstration sautee - voir l'output de reference dans le notebook version main.")


INFLECT-NANO-V1 - TTS ULTRA-LEGER
Dependance manquante (No module named 'torch'). Requiert : torch, soundfile, g2p_en, huggingface_hub.
Inflect-Nano n'est pas installe - demonstration sautee (graceful).


### Interpretation : le compromis taille vs qualite

L'exécution locale (CPU, RTF < 1) confirme les deux proprietes cles d'Inflect-Nano :

1. **Vitesse extreme** : grace a l'architecture non-autoregressive (FastSpeech), toutes les frames mel sont predites en parallele. Sur CPU, *runtime machine-dep* : la generation est plus rapide que la lecture - un cas rare pour un TTS neuronal.

2. **Taille minimale** : 4,63M de paramètres tiennent dans ~49 MB. Aucun GPU requis. Ce profil correspond aux contraintes embarquees (IoT, edge) ou la VRAM et l'energie sont budget nul.

**Mais le cout est visible** : la voix est manifestement robotique, monophone (pas de variation de timbre), limitee a l'anglais, et incapable de reproduire une identite vocale. La where Kokoro investit dans le naturel et la diversite, Inflect-Nano investit uniquement dans la fonctionnalite nue.

### Ou se situe la frontiere ?

| Objectif prioritaire | Modèle recommande | Pourquoi |
|----------------------|-------------------|----------|
| **Production multi-langue, naturel** | Kokoro (82M) ou XTTS | Compromis taille/qualite raisonnable |
| **Qualite maximale, cout illimite** | OpenAI TTS / FishAudio S2-Pro | API cloud, voix expressives |
| **Enonce minimal, zero ressource** | Inflect-Nano (4,63M) | Frontiere inferieure du TTS neuronal |
| **Clonage vocal zero-shot** | XTTS v2 | Non faisable en dessous de ~400M |

> **Lecon pedagogique** : un TTS "fonctionnel" demande un minimum structural (acoustique + vocodeur, ~5M). En dessous, on produit encore du bruit structure mais plus de la parole. Inflect-Nano se place juste au-dessus de ce seuil, ce qui en fait un excellent cas d'étude pour comprendre l'anatomie minimale d'un système TTS. La serie continue vers Chatterbox (02-1) et XTTS (02-2) pour observer comment les ~460M supplementaires se traduisent en naturel, expressivite et clonage.

***

*Source : modèle [owensong/Inflect-Nano-v1](https://huggingface.co/owensong/Inflect-Nano-v1) (Apache-2.0). Comparaison effectuee en exécution locale CPU.*
